# Imports

In [1]:
import sys
import numpy as np
import torch
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
sys.path.append('../..')
from torch import Tensor
from src.commands.graph_learning.main import Config, Trainer, HeteroData, Graph, Dict, Any, List
from typing import Tuple
from definitions import EXPERIMENTS_DIR, ROOT_DIR
from PIL import ImageColor
from torch_scatter import scatter
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
    confusion_matrix
)
from umap import UMAP
from sklearn.preprocessing import StandardScaler

In [2]:
px.defaults.template = 'seaborn'

In [3]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Utils

In [19]:
def load_data(
    experiment_id : str,
    config_id : str
) -> Tuple[HeteroData, Graph]:

    config = Config(experiment=experiment_id)
    data, graph = config.training_config.configs[config_id].load_data(return_raw=True)

    trainer = Trainer(
        experiment=experiment_id, 
        config=config.training_config,
        configs=None
    )

    # data, graph = trainer.load_data(config=exper_config, return_raw=True)
    split = trainer.create_or_load_split(data=data)

    data['Article'].split = split

    return data, graph

In [5]:
def load_history(experiment_id : str) -> pd.DataFrame:

    histories_dir = os.path.join(EXPERIMENTS_DIR, experiment_id, 'histories')
    histories = []

    for filename in os.listdir(histories_dir):

        if "_final" in filename:
            continue

        history_path = os.path.join(histories_dir, filename)
        history = pd.read_csv(history_path)
        history['trial'] = int(os.path.splitext(filename)[0][-1])
        histories.append(history)

    return pd.concat(histories)

In [6]:
def aggregate_history(history : pd.DataFrame) -> pd.DataFrame:

    history = history.groupby(by=['epoch', 'phase', 'config', 'trial']) \
        .mean().reset_index()

    history_mean = history \
        .drop(columns = 'trial') \
        .groupby(['epoch','phase','config']).mean() \
        .add_suffix('_mean')

    history_std = history \
        .drop(columns = 'trial') \
        .groupby(['epoch','phase','config']).std() \
        .add_suffix('_std')

    return history_mean.join(other = history_std).reset_index()

In [7]:
def load_results(experiment_id : str) -> Dict[Tuple[str, int], Any]:

    results_dir = os.path.join(EXPERIMENTS_DIR, experiment_id, 'results')
    results = {}

    for filename in os.listdir(results_dir):

        if "_final" in filename:
            continue

        config_id = filename.split('_')[1]
        trial = int(os.path.splitext(filename)[0][-1])
        results_path = os.path.join(results_dir, filename)
        results[(config_id, trial)] = torch.load(results_path, weights_only=False)

    return results

In [8]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> go.Figure:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['config'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and config == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

    return fig

In [9]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [10]:
def test_articles_subset(
    data : HeteroData,
    min_p : float = 0.5
) -> Tensor:
    
    fold = 0
    split = data.split
    mask = split != fold

    entity_ids, articles_ids = data['Entity', 'APPEARIN', 'Article'].edge_index
    train_entities_ids = torch.unique(entity_ids[mask[articles_ids]])

    p = scatter(
        torch.isin(entity_ids, train_entities_ids).float(),
        articles_ids,
        dim=0,
        dim_size=data['Article'].x.size(0),
        reduce='mean'
    )

    p = p[~mask]

    node_ids = torch.arange(len(p)).to(p.device)

    return node_ids[p >= min_p]

In [11]:
def get_label_wise_metrics(
    label_name : str,
    graph : HeteroData,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    data = graph['Article']

    n_labels = data[f'{label_name}']
    n_labels = n_labels.size(-1)

    for label in range(n_labels):
            
        for (expr,trial),result in results.items():

            y = data[f'{label_name}'][data.split == trial]

            if label_name not in result['articles_labels']:
                continue

            y_hat = result['articles_labels'][label_name][data.split == trial].sigmoid()
            y_hard = y_hat.gt(0.5).long()

            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label], zero_division=0),
                "precision" : precision_score(y[:,label], y_hard[:,label], zero_division=0),
                "recall" : recall_score(y[:,label], y_hard[:,label], zero_division=0),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label]),
                "TN" : confusion_matrix(y[:,label], y_hard[:,label], labels=[0,1])[0,0],
                "FP" : confusion_matrix(y[:,label], y_hard[:,label],labels=[0,1])[0,1],
                "FN" : confusion_matrix(y[:,label], y_hard[:,label],labels=[0,1])[1,0],
                "TP" : confusion_matrix(y[:,label], y_hard[:,label],labels=[0,1])[1,1]
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [12]:
def get_row_wise_metrics(
    label : str,
    data : HeteroData,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        if label_name not in result['articles_labels']:
            continue

        y = data[f'y_{label}'][data.split == trial]
        y_hat = result['articles_labels'][label][data.split == trial].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        if label == 'lab':
            y = y[:,:-1]
            y_hat = y_hat[:,:-1]
            y_hard = y_hard[:,:-1]

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro', zero_division=0),
            "precision" : precision_score(y, y_hard , average='macro', zero_division=0),
            "recall" : recall_score(y, y_hard , average='macro', zero_division=0),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [13]:
def aggregate_label_wise_metrics(
    data : HeteroData,
    label_wise_metrics : pd.DataFrame, 
    config_id : str,
    label : str,
    label_names : List[str]
) -> pd.DataFrame:

    labs2count = dict(zip(label_names, data['Article'][label].sum(dim=0).tolist()))
    
    summary = label_wise_metrics \
        .query(f"expr == '{config_id}'") \
        .drop(columns='trial') \
        .groupby(by=['team', 'expr']) \
        .mean() \
        .reset_index()

    summary['Num. Articles'] = summary['team'].map(labs2count.get)
    summary = summary.sort_values(by='Num. Articles', ascending=False)
    return summary

In [14]:
def umap_visualization(
    results: Dict[str, Any],
    id : Tuple[str, int],
    data: HeteroData,
    label: str,
    labels_names: List[str],
) -> go.Figure:

    mask = (data['Article'].split == id[1]) if isinstance(id, int) else torch.ones_like(data['Article'].split).bool()
    X = results[id]['articles_embeddings'][mask,:].numpy()
    y = data['Article'][label][mask,:].argmax(dim=-1).cpu().numpy()

    scaler = StandardScaler()
    X = scaler.fit_transform(X=X)
    X2d = UMAP(n_components=2).fit_transform(X=X)

    df = pd.DataFrame({
        'UMAP 1': X2d[:, 0],
        'UMAP 2': X2d[:, 1],
        'label': [labels_names[i] for i in y],
        'article_id': np.arange(len(y)),
    })

    fig = px.scatter(
        df,
        x='UMAP 1',
        y='UMAP 2',
        color='label',
        category_orders={'label': labels_names},
        hover_data={'article_id': True, 'UMAP 1': ':.2f', 'UMAP 2': ':.2f'},
        title=f'UMAP projection of article embeddings — colored by "{label.strip('y_')}"',
        opacity=0.7,
    )

    fig.update_traces(marker=dict(size=5, line=dict(width=0)))
    fig.update_layout(
        width=900,
        height=700,
        template='plotly_white',
        legend_title_text=label.strip('y_').capitalize(),
        legend=dict(itemsizing='constant'),
    )

    fig.show()

    return fig

# Load Data, Results & History

In [20]:
data, graph = load_data(
    experiment_id='health_care_v3', 
    config_id='baseline'
)

In [22]:
history = load_history(experiment_id='health_care_v3')
history.head()

,epoch,phase,loss,bec_loss_llm:deepseek-chat__exact_accuracy,bec_loss_llm:deepseek-chat__accuracy,bec_loss_llm:deepseek-chat__macro_f1_score,bec_loss_llm:deepseek-chat__macro_precision,bec_loss_llm:deepseek-chat__macro_recall,bec_loss_llm:deepseek-chat,bec_loss_lab__exact_accuracy,bec_loss_lab__accuracy,bec_loss_lab__macro_f1_score,bec_loss_lab__macro_precision,bec_loss_lab__macro_recall,bec_loss_lab,config,trial
0,0,train,0.347216,0.0,0.512155,0.041817,0.028420,0.470223,0.345688,0.000000,0.402643,0.028950,0.015779,0.440161,0.348744,baseline,2
1,0,train,0.331397,0.0,0.809005,0.024507,0.018763,0.182591,0.332606,0.000000,0.810884,0.011437,0.012037,0.110801,0.330187,baseline,2
2,0,train,0.306542,0.0,0.869205,0.007702,0.004472,0.125351,0.311031,0.000000,0.925215,0.011235,0.010228,0.068580,0.302053,baseline,2
3,0,train,0.261137,0.0,0.933404,0.002568,0.004458,0.018473,0.270018,0.053633,0.957747,0.002907,0.004665,0.023932,0.252256,baseline,2
4,0,train,0.190853,0.0,0.953193,0.000183,0.000092,0.010256,0.203737,0.405310,0.979154,0.000000,0.000000,0.000000,0.177970,baseline,2


In [23]:
aggregated_history = aggregate_history(history)
aggregated_history.head()

,epoch,phase,config,loss_mean,bec_loss_llm:deepseek-chat__exact_accuracy_mean,bec_loss_llm:deepseek-chat__accuracy_mean,bec_loss_llm:deepseek-chat__macro_f1_score_mean,bec_loss_llm:deepseek-chat__macro_precision_mean,bec_loss_llm:deepseek-chat__macro_recall_mean,bec_loss_llm:deepseek-chat_mean,...,bec_loss_llm:deepseek-chat__macro_f1_score_std,bec_loss_llm:deepseek-chat__macro_precision_std,bec_loss_llm:deepseek-chat__macro_recall_std,bec_loss_llm:deepseek-chat_std,bec_loss_lab__exact_accuracy_std,bec_loss_lab__accuracy_std,bec_loss_lab__macro_f1_score_std,bec_loss_lab__macro_precision_std,bec_loss_lab__macro_recall_std,bec_loss_lab_std
0,0,train,baseline,0.039245,0.247646,0.968883,0.181244,0.322848,0.144213,0.046250,...,0.004827,0.006654,0.003745,0.000316,0.004435,0.000306,0.002246,0.003457,0.001789,0.000267
1,0,train,labs,0.016462,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.004092,0.000321,0.001577,0.004123,0.001184,0.000131
2,0,val,baseline,0.037905,0.268488,0.968942,0.236394,0.385088,0.191080,0.044984,...,0.018626,0.003430,0.020670,0.001150,0.005539,0.000359,0.018044,0.014773,0.018722,0.000803
3,0,val,labs,0.015446,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.021476,0.000615,0.017210,0.012993,0.015353,0.000511
4,1,train,baseline,0.026302,0.345850,0.977643,0.354945,0.558876,0.288208,0.030729,...,0.006354,0.008404,0.005773,0.000075,0.005125,0.000214,0.003301,0.003208,0.003072,0.000259


In [25]:
results = load_results(experiment_id='health_care_v3')
# results.pop(('labs', 0))
# results.pop(('labs', 1))
print(list(results.keys()))

[('labs', 3), ('baseline', 1), ('labs', 0), ('baseline', 0), ('labs', 1), ('baseline', 2), ('labs', 2), ('baseline', 3)]


# Learning Curves

In [26]:
fig = line_plot(aggregated_history, 'loss')

In [27]:
fig = line_plot(aggregated_history, 'bec_loss_llm:deepseek-chat')

In [29]:
fig = line_plot(aggregated_history, 'bec_loss_lab')

In [30]:
fig = line_plot(aggregated_history, 'bec_loss_llm:deepseek-chat__macro_f1_score')

In [31]:
fig = line_plot(aggregated_history, 'bec_loss_lab__macro_f1_score')

In [33]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics(
    'lab', 
    data, 
    results, 
    data.raw_metadata['labels']['lab']
)

display_mean_std_df(label_wise_metrics_agg)

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_true. ROC AUC score is not defined in that case.

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present 

,expr,precision,FN,auc,recall,TN,FP,accuracy,TP,f1_score
0,baseline,33.77% ± 2.43%,4763.33% ± 131.90%,88.46% ± 1.03%,18.47% ± 1.15%,290236.11% ± 105.01%,2169.44% ± 140.03%,97.68% ± 0.05%,2181.11% ± 75.44%,23.00% ± 1.24%
1,labs,30.98% ± 1.68%,4957.22% ± 126.24%,86.01% ± 0.52%,14.62% ± 0.73%,290473.33% ± 92.07%,1932.22% ± 97.04%,97.70% ± 0.01%,1987.22% ± 61.33%,18.82% ± 0.69%


In [56]:
labels = data.raw_metadata['labels']['lab']

thorax_idx = labels.index("thorax-umr")
all_idx = labels.index('thorax-umr/thorax/chu_de_nantes')

print(thorax_idx, all_idx)

Y_hat = results[('baseline', 3)]['articles_labels']['lab'].sigmoid().gt(0.5).float()
M = (data['Article']['lab'][:,thorax_idx] == 1).float()
(Y_hat[:,all_idx] * M).sum()

0 44


tensor(342.)

In [45]:
results[('baseline', 1)]['articles_labels']['lab']

tensor([[  1.1923,  -6.7472, -11.8221,  ..., -11.4202, -10.7889,   0.7133],
        [ -2.3728,  -1.6052,  -4.5403,  ...,  -7.0025,  -7.3895,  -0.2339],
        [ -2.9324,  -4.3910,  -4.1572,  ...,  -6.5984,  -7.0485,   0.9770],
        ...,
        [ -5.5823,  -2.8624,  -4.8024,  ...,  -8.8918,  -7.2751,  -9.4445],
        [ -8.2383,  -6.4428,  -2.7239,  ...,  -5.8596,  -9.1094, -12.1766],
        [ -5.6046,  -3.0255,  -6.9720,  ...,  -6.1275,  -7.9721,  -6.0639]])

In [35]:
aggregate_label_wise_metrics(
    data=data,
    label_wise_metrics=label_wise_metrics,
    config_id='baseline',
    label='lab',
    label_names=data.raw_metadata['labels']['lab']
)

,team,expr,accuracy,f1_score,precision,recall,auc,TN,FP,FN,TP,Num. Articles
41,thorax-umr/thorax/chu_de_nantes,baseline,0.776181,0.343123,0.382646,0.311938,0.719294,2148.50,284.00,386.00,175.00,2244.0
4,chu_de_nantes,baseline,0.834892,0.327410,0.396979,0.279407,0.743702,2378.75,183.75,310.50,120.50,1724.0
27,oniris,baseline,0.924754,0.568230,0.639426,0.511615,0.888865,2619.75,83.75,141.50,148.50,1160.0
8,crci2na,baseline,0.912561,0.350549,0.441861,0.291342,0.853234,2661.00,89.50,172.25,70.75,972.0
6,cr2ti,baseline,0.947720,0.257313,0.380761,0.195584,0.836896,2809.75,43.50,113.00,27.25,561.0
22,ls2n,baseline,0.963421,0.465449,0.613008,0.377130,0.939862,2835.25,30.25,79.25,48.75,512.0
40,thorax-umr,baseline,0.936530,0.045499,0.061946,0.036452,0.782733,2799.00,69.25,120.75,4.50,501.0
3,ceisam,baseline,0.978704,0.632504,0.775971,0.538762,0.967533,2874.50,16.00,47.75,55.25,412.0
10,gem,baseline,0.982128,0.699949,0.750752,0.658507,0.984717,2877.50,21.00,32.50,62.50,380.0
1,bia,baseline,0.977869,0.548322,0.692855,0.454191,0.961698,2887.25,17.75,48.50,40.00,354.0


In [26]:
aggregate_label_wise_metrics(
    data=data,
    label_wise_metrics=label_wise_metrics,
    config_id='labs',
    label='y_lab',
    label_names=data.raw_metadata['labels']['lab']
)

,team,expr,accuracy,f1_score,precision,recall,auc,TN,FP,FN,TP,Num. Articles
4,chu_de_nantes,labs,0.851153,0.342979,0.439134,0.282383,0.770971,2431.0,148.5,297.0,116.5,1724.0
27,oniris,labs,0.927665,0.578561,0.629766,0.535231,0.891206,2627.0,87.5,129.0,149.5,1160.0
8,crci2na,labs,0.915469,0.407569,0.477573,0.357596,0.880709,2653.0,95.5,157.5,87.0,972.0
6,cr2ti,labs,0.947377,0.245439,0.365876,0.184662,0.847683,2810.0,44.5,113.0,25.5,561.0
22,ls2n,labs,0.964751,0.448843,0.630645,0.349950,0.921219,2844.5,25.5,80.0,43.0,512.0
40,thorax-umr,labs,0.941196,0.062727,0.092157,0.047544,0.757581,2811.0,58.0,118.0,6.0,501.0
3,ceisam,labs,0.977781,0.614330,0.788359,0.511358,0.968401,2873.5,14.5,52.0,53.0,412.0
10,gem,labs,0.983628,0.720278,0.768251,0.678030,0.987139,2880.5,19.0,30.0,63.5,380.0
1,bia,labs,0.977447,0.537450,0.765909,0.431373,0.954971,2886.0,13.5,54.0,39.5,354.0
25,mip,labs,0.977280,0.517103,0.734336,0.406002,0.949578,2888.5,13.0,55.0,36.5,334.0


In [27]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics(
    'llm:deepseek-chat', 
    data, 
    results, 
    data.raw_metadata['labels']['subdomain']
)

display_mean_std_df(label_wise_metrics_agg)

,expr,FN,precision,auc,f1_score,FP,TP,accuracy,TN,recall
0,baseline,6568.46% ± 198.39%,48.74% ± 1.88%,91.87% ± 0.72%,34.55% ± 1.09%,2907.69% ± 64.24%,4825.00% ± 113.90%,96.83% ± 0.07%,285048.85% ± 139.41%,28.37% ± 0.98%
1,domains,6646.54% ± 152.08%,47.57% ± 2.11%,91.71% ± 0.56%,31.64% ± 0.36%,2940.77% ± 72.17%,4746.92% ± 80.69%,96.80% ± 0.03%,285015.77% ± 53.48%,25.63% ± 0.33%


In [29]:
summary = aggregate_label_wise_metrics(
    data=data,
    label_wise_metrics=label_wise_metrics,
    config_id='baseline',
    label='llm:deepseek-chat',
    label_names=data.raw_metadata['labels']['subdomain']
)

summary

,team,expr,accuracy,f1_score,precision,recall,auc,TN,FP,FN,TP,Num. Articles
57,real-world_data_epidemiology_&_biostatistics,baseline,0.877067,0.768722,0.770612,0.767122,0.934897,2013.00,182.50,185.50,612.50,3192.0
14,clinical_research_&_ux_methodologies,baseline,0.836898,0.442033,0.509552,0.391045,0.799158,2311.50,186.25,302.00,193.75,1983.0
13,clinical_oncology_&_nuclear_medicine,baseline,0.882496,0.560846,0.598453,0.529056,0.877743,2417.00,151.25,200.50,224.75,1701.0
17,comprehensive_oncology_&_radiotherapy,baseline,0.890513,0.484479,0.554637,0.430623,0.859398,2511.75,124.00,203.75,154.00,1431.0
39,medico-economic_analysis_&_innovation,baseline,0.867881,0.268506,0.366028,0.212276,0.737775,2525.50,125.50,270.00,72.50,1370.0
...,...,...,...,...,...,...,...,...,...,...,...,...
26,glycochemistry_&_dna_repair_mechanisms,baseline,0.995323,0.000000,0.000000,0.000000,0.852433,2979.50,0.00,14.00,0.00,56.0
19,dna_repair_&_environmental_toxicity,baseline,0.996158,0.000000,0.000000,0.000000,0.870441,2982.00,0.00,11.50,0.00,46.0
34,marine_metabolomics_&_antibacterial_properties,baseline,0.996576,0.000000,0.000000,0.000000,0.957126,2983.25,0.00,10.25,0.00,41.0
37,market_access_&_regulatory_strategy,baseline,0.996994,0.000000,0.000000,0.000000,0.910423,2984.50,0.00,9.00,0.00,36.0


In [30]:
summary[summary['f1_score'] >= 0.4]

,team,expr,accuracy,f1_score,precision,recall,auc,TN,FP,FN,TP,Num. Articles
57,real-world_data_epidemiology_&_biostatistics,baseline,0.877067,0.768722,0.770612,0.767122,0.934897,2013.00,182.50,185.50,612.50,3192.0
14,clinical_research_&_ux_methodologies,baseline,0.836898,0.442033,0.509552,0.391045,0.799158,2311.50,186.25,302.00,193.75,1983.0
13,clinical_oncology_&_nuclear_medicine,baseline,0.882496,0.560846,0.598453,0.529056,0.877743,2417.00,151.25,200.50,224.75,1701.0
17,comprehensive_oncology_&_radiotherapy,baseline,0.890513,0.484479,0.554637,0.430623,0.859398,2511.75,124.00,203.75,154.00,1431.0
46,oncology_&_multi-omics,baseline,0.914482,0.577050,0.637224,0.528306,0.903582,2562.50,99.75,156.25,175.00,1325.0
9,cardiometabolic_pathophysiology_&_target_ident...,baseline,0.937698,0.662163,0.699009,0.630358,0.936634,2624.00,79.25,107.25,183.00,1161.0
12,chronic_inflammatory_diseases_&_biomarkers,baseline,0.922582,0.434216,0.542171,0.363897,0.883356,2672.75,75.25,156.50,89.00,982.0
28,healthcare_workflow_&_ux_evaluation,baseline,0.951061,0.566031,0.633194,0.515315,0.952209,2751.50,55.50,91.00,95.50,746.0
38,materials_science_&_digital_twins,baseline,0.981209,0.835912,0.882329,0.795729,0.989419,2794.00,19.25,37.00,143.25,721.0
8,cardiometabolic_multi-omics_&_pharmacology,baseline,0.951812,0.521610,0.622077,0.449825,0.921221,2770.25,47.75,96.50,79.00,702.0


In [31]:
def save_fig(fig : go.Figure, filename : str) -> None:
    fig.write_image(os.path.join('.', filename))

In [32]:
fig = umap_visualization(
    results=results,
    id=('baseline', 0),
    data=data,
    label='y_lab',
    labels_names=data.raw_metadata['labels']['lab']
)

# save_fig(fig, 'vis.pdf')